In [1]:
import numpy as np

class RNN:
    def __init__(self, inp, hid, out, lr=0.001, T=5):
        self.inp, self.hid, self.out, self.lr, self.T = inp, hid, out, lr, T
        self.Wxh = np.random.randn(hid, inp) * 0.01
        self.Whh = np.random.randn(hid, hid) * 0.01
        self.Why = np.random.randn(out, hid) * 0.01
        self.bh, self.by = np.zeros((hid,1)), np.zeros((out,1))

    def forward(self, X):
        h = np.zeros((self.hid,1)); self.h, self.y = {-1:h}, {}
        for t in range(self.T):
            h = np.tanh(self.Wxh@X[t] + self.Whh@self.h[t-1] + self.bh)
            self.h[t], self.y[t] = h, self.Why@h + self.by
        return self.y

    def backward(self, X, Tgt):
        dWxh=dWhh=dWhy=0; dbh=dby=0; dhn=np.zeros((self.hid,1))
        for t in reversed(range(self.T)):
            dy = self.y[t]-Tgt[t]; dWhy+=dy@self.h[t].T; dby+=dy
            dh = self.Why.T@dy + dhn; dh_raw=(1-self.h[t]**2)*dh
            dbh+=dh_raw; dWxh+=dh_raw@X[t].T; dWhh+=dh_raw@self.h[t-1].T
            dhn=self.Whh.T@dh_raw
        for d in [dWxh,dWhh,dWhy,dbh,dby]: np.clip(d,-5,5,out=d)
        for p,d in zip([self.Wxh,self.Whh,self.Why,self.bh,self.by],
                       [dWxh,dWhh,dWhy,dbh,dby]): p-=self.lr*d

    def train(self, data, labels, epochs=50):
        for e in range(epochs):
            loss=0
            for X,Tgt in zip(data,labels):
                out=self.forward(X); self.backward(X,Tgt)
                loss+=np.sum((out[self.T-1]-Tgt[self.T-1])**2)/2
            if e%10==0: print(f"Epoch {e}, Loss:{loss:.4f}")

# Example
if __name__=="__main__":
    T,inp,hid,out=5,3,4,2
    data=[np.random.randn(T,inp,1)*0.1 for _ in range(100)]
    labels=[np.random.randn(T,out,1)*0.1 for _ in range(100)]
    rnn=RNN(inp,hid,out); rnn.train(data,labels,epochs=50)

Epoch 0, Loss:1.0461
Epoch 10, Loss:1.0469
Epoch 20, Loss:1.0469
Epoch 30, Loss:1.0469
Epoch 40, Loss:1.0469


In [8]:
import numpy as np

class RecurrentNeuralNetwork:
    def __init__(self, input_dim, hidden_dim, output_dim, lr=0.001, time_steps=5):
        self.input_dim, self.hidden_dim, self.output_dim = input_dim, hidden_dim, output_dim
        self.lr, self.time_steps = lr, time_steps

        # Weight initialization
        self.Wxh = np.random.randn(hidden_dim, input_dim) * 0.01
        self.Whh = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.Why = np.random.randn(output_dim, hidden_dim) * 0.01
        self.bh = np.zeros((hidden_dim, 1))
        self.by = np.zeros((output_dim, 1))

    def forward(self, inputs):
        h = np.zeros((self.hidden_dim, 1))
        self.h_states, self.outputs = {-1: h}, {}
        for t in range(self.time_steps):
            h = np.tanh(self.Wxh @ inputs[t] + self.Whh @ self.h_states[t-1] + self.bh)
            y = self.Why @ h + self.by
            self.h_states[t], self.outputs[t] = h, y
        return self.outputs

    def backward(self, inputs, targets):
        dWxh, dWhh, dWhy = np.zeros_like(self.Wxh), np.zeros_like(self.Whh), np.zeros_like(self.Why)
        dbh, dby = np.zeros_like(self.bh), np.zeros_like(self.by)
        dh_next = np.zeros((self.hidden_dim, 1))

        for t in reversed(range(self.time_steps)):
            dy = self.outputs[t] - targets[t]
            dWhy += dy @ self.h_states[t].T
            dby += dy
            dh = self.Why.T @ dy + dh_next
            dh_raw = (1 - self.h_states[t] ** 2) * dh
            dbh += dh_raw
            dWxh += dh_raw @ inputs[t].T
            dWhh += dh_raw @ self.h_states[t-1].T
            dh_next = self.Whh.T @ dh_raw

        # Gradient clipping
        for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
            np.clip(dparam, -5, 5, out=dparam)

        # Update weights
        for param, dparam in zip([self.Wxh, self.Whh, self.Why, self.bh, self.by],
                                 [dWxh, dWhh, dWhy, dbh, dby]):
            param -= self.lr * dparam

    def train(self, data, labels, epochs=50):
        for epoch in range(epochs):
            loss = 0
            for inputs, targets in zip(data, labels):
                outputs = self.forward(inputs)
                self.backward(inputs, targets)
                loss += np.sum((outputs[self.time_steps-1] - targets[self.time_steps-1])**2) / 2
            if epoch % 10 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

# Example usage
if __name__ == "__main__":
    time_steps, input_dim, hidden_dim, output_dim = 5, 3, 4, 2
    data = [np.random.randn(time_steps, input_dim, 1) * 0.1 for _ in range(100)]
    labels = [np.random.randn(time_steps, output_dim, 1) * 0.1 for _ in range(100)]
    rnn = RecurrentNeuralNetwork(input_dim, hidden_dim, output_dim)
    rnn.train(data, labels, epochs=50)

Epoch 0, Loss: 0.9018
Epoch 10, Loss: 0.9021
Epoch 20, Loss: 0.9022
Epoch 30, Loss: 0.9022
Epoch 40, Loss: 0.9022


In [10]:
import numpy as np

class RecurrentNeuralNetwork:
    def __init__(self, input_dim, hidden_dim, output_dim, learning_rate=0.001, time_steps=5):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.learning_rate = learning_rate
        self.time_steps = time_steps

        # Initialize weights
        self.Wxh = np.random.randn(hidden_dim, input_dim) * 0.01  # Input to hidden
        self.Whh = np.random.randn(hidden_dim, hidden_dim) * 0.01  # Hidden to hidden
        self.Why = np.random.randn(output_dim, hidden_dim) * 0.01  # Hidden to output
        self.bh = np.zeros((hidden_dim, 1))
        self.by = np.zeros((output_dim, 1))

    def forward(self, inputs):
        """Forward pass through time."""
        h = np.zeros((self.hidden_dim, 1))  # Initial hidden state
        self.h_states = {-1: h}
        self.outputs = {}

        for t in range(self.time_steps):
            h = np.tanh(np.dot(self.Wxh, inputs[t]) +
                        np.dot(self.Whh, self.h_states[t-1]) + self.bh)
            y = np.dot(self.Why, h) + self.by
            self.h_states[t] = h
            self.outputs[t] = y

        return self.outputs

    def backward(self, inputs, targets):
        """Backward pass using Backpropagation Through Time (BPTT)."""
        dWxh, dWhh, dWhy = np.zeros_like(self.Wxh), np.zeros_like(self.Whh), np.zeros_like(self.Why)
        dbh, dby = np.zeros_like(self.bh), np.zeros_like(self.by)
        dh_next = np.zeros((self.hidden_dim, 1))

        for t in reversed(range(self.time_steps)):
            dy = self.outputs[t] - targets[t]
            dWhy += np.dot(dy, self.h_states[t].T)
            dby += dy

            dh = np.dot(self.Why.T, dy) + dh_next
            dh_raw = (1 - self.h_states[t] ** 2) * dh  # Derivative of tanh
            dbh += dh_raw
            dWxh += np.dot(dh_raw, inputs[t].T)
            dWhh += np.dot(dh_raw, self.h_states[t-1].T)
            dh_next = np.dot(self.Whh.T, dh_raw)

        # Gradient clipping
        for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
            np.clip(dparam, -5, 5, out=dparam)

        # Update weights
        for param, dparam in zip([self.Wxh, self.Whh, self.Why, self.bh, self.by],
                                 [dWxh, dWhh, dWhy, dbh, dby]):
            param -= self.learning_rate * dparam

    def train(self, data, labels, epochs=100):
        """Train the network over multiple epochs."""
        for epoch in range(epochs):
            loss = 0
            for inputs, targets in zip(data, labels):
                outputs = self.forward(inputs)
                self.backward(inputs, targets)
                loss += np.sum((outputs[self.time_steps-1] - targets[self.time_steps-1])**2) / 2

            if epoch % 10 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")


# Example usage
if __name__ == "__main__":
    time_steps = 5
    input_dim = 3
    hidden_dim = 4
    output_dim = 2

    # Normalize input data
    data = [np.random.randn(time_steps, input_dim, 1) * 0.1 for _ in range(100)]
    labels = [np.random.randn(time_steps, output_dim, 1) * 0.1 for _ in range(100)]

    rnn = RecurrentNeuralNetwork(input_dim, hidden_dim, output_dim)
    rnn.train(data, labels, epochs=50)

Epoch 0, Loss: 1.1038
Epoch 10, Loss: 1.0971
Epoch 20, Loss: 1.0971
Epoch 30, Loss: 1.0971
Epoch 40, Loss: 1.0971
